<a href="https://colab.research.google.com/github/meghansn/mental-health-llm-pipeline/blob/rag-disorders/RAG_Diagnosis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.cloud import bigquery

import vertexai
from vertexai.generative_models import GenerativeModel

import json

In [ ]:
from google.colab import auth

auth.authenticate_user()

In [ ]:
#initialize vertex AI
PROJECT_ID = "mental-health-llm-pip"
LOCATION = "us-central1"

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION
)

In [ ]:
#create clients
client = bigquery.Client(project=PROJECT_ID)

model = GenerativeModel("gemini-2.5-flash")

In [ ]:
#sanity check
client

In [ ]:
model

In [ ]:
query = """
WITH random_patient AS (
    SELECT patient_id
    FROM `mental-health-llm-pip.patient_insights.dim_patients`
    ORDER BY RAND()
    LIMIT 1
)

SELECT
    dp.*,
    fs.session_date, -- Added session_date from fact_sessions (fs)
    fsr.*
FROM random_patient rp
JOIN `mental-health-llm-pip.patient_insights.dim_patients` dp
    ON rp.patient_id = dp.patient_id
JOIN `mental-health-llm-pip.patient_insights.fact_sessions` fs
    ON dp.patient_id = fs.patient_id
JOIN `mental-health-llm-pip.patient_insights.fact_sessions_redacted` fsr
    ON fs.session_id = fsr.session_id
ORDER BY fs.session_date
"""

In [ ]:
#put in dataframe
df = client.query(query).to_dataframe()

In [ ]:
df

In [ ]:
#initiate empty longitudinal list of symptoms
longitudinal_symptoms = []

In [ ]:
#iterate through symptom row
for index, row in df.iterrows():
  #place session date
  longitudinal_symptoms.append(f"Session Date: {row['session_date']}")
  #fetch sessions
  longitudinal_symptoms.append(f"Symptoms: {row['extracted_symptoms']}")
longitudinal_symptoms

In [ ]:
patient_history = "\n\n".join(longitudinal_symptoms)

In [ ]:
patient_history

In [ ]:
from vertexai.language_models import TextEmbeddingModel
embedding_model = TextEmbeddingModel.from_pretrained(
    "text-embedding-004"
)
#get embeddings
query_embedding = embedding_model.get_embeddings([patient_history])[0].values

In [ ]:
# Load all disorder embeddings
query = """
SELECT
    disorder_id,
    disorder_name,
    retrieval_text,
    embedding
FROM `mental-health-llm-pip.mental_health.disorder_embeddings`
"""

disorders_df = client.query(query).to_dataframe()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

disorders_df["similarity"] = disorders_df["embedding"].apply(
    lambda x: cosine_similarity(
        [query_embedding],
        [x]
    )[0][0]
)

In [ ]:
top_disorders = (
    disorders_df
    .sort_values("similarity", ascending=False)
    .head(5)
)

top_disorders[["disorder_name", "similarity"]]

In [ ]:
rag_context = ""

In [ ]:
for index, row in top_disorders.iterrows():
    rag_context += f"Disorder: {row['disorder_name']}\n"
    rag_context += f"{row['retrieval_text']}\n\n"

In [ ]:
print(rag_context)

In [ ]:
prompt = f"""
You are a psychiatrist.

Below is a patient's longitudinal symptom history.

{patient_history}

Below are the most relevant DSM disorders retrieved from the knowledge base.

{rag_context}

Using ONLY the retrieved DSM information, determine the most likely diagnosis.

Return valid JSON:

{{
  "primary_diagnosis": "",
  "confidence": "",
  "reasoning": "",
  "supporting_symptoms": [],
  "differential_diagnoses": []
}}
"""

In [ ]:
llm = GenerativeModel("gemini-2.5-flash")

In [ ]:
response = llm.generate_content(prompt)

In [ ]:
print(response.text)

In [ ]:
import json

diagnosis = json.loads(response.text.strip("```json").strip("```"))

In [ ]:
diagnosis